# Pipeline Orquestador — Embalse Guatapé
Lanza un SageMaker Pipeline con 5 steps en secuencia:
1. Ingesta
2. Procesamiento
3. Entrenamiento ARIMA
4. Inferencia ARIMA
5. Entrenamiento LSTM (incluye inferencia)

**Requisito:** ejecutar desde una instancia de notebook de SageMaker con rol `SageMakerExecutionRole`.

In [ ]:
# ── Instalación ───────────────────────────────────────────────────────────────
import subprocess, sys
subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'sagemaker', '--upgrade', '-q'])

In [ ]:
import boto3
import sagemaker
from sagemaker.workflow.pipeline import Pipeline
from sagemaker.workflow.steps import ProcessingStep
from sagemaker.processing import ScriptProcessor, ProcessingInput, ProcessingOutput
from sagemaker.workflow.pipeline_context import PipelineSession

print(f'SageMaker SDK: {sagemaker.__version__}')

In [ ]:
# ── Configuración central ─────────────────────────────────────────────────────
BUCKET         = 'embalses-colombia'
ROLE           = 'arn:aws:iam::025627370678:role/SageMakerExecutionRole'
REGION         = 'us-east-1'
PIPELINE_NAME  = 'pipeline-guatape'
IMAGE_URI      = '683313688378.dkr.ecr.us-east-1.amazonaws.com/sagemaker-scikit-learn:1.2-1-cpu-py3'
INSTANCE_TYPE  = 'ml.t3.medium'

# Prefijos S3
SCRIPTS_PREFIX = f's3://{BUCKET}/scripts'
CURATED_S3     = f's3://{BUCKET}/data/curated/embalse_guatape/'
MODEL_ARIMA_S3 = f's3://{BUCKET}/models/arima/latest/'
PREDS_ARIMA_S3 = f's3://{BUCKET}/predictions/arima/'

session        = PipelineSession(boto_session=boto3.Session(region_name=REGION))
print('Configuración OK')

In [ ]:
# ── Procesador base ───────────────────────────────────────────────────────────
def make_processor():
    return ScriptProcessor(
        image_uri=IMAGE_URI,
        command=['python3'],
        instance_type=INSTANCE_TYPE,
        instance_count=1,
        role=ROLE,
        sagemaker_session=session,
    )

In [ ]:
# ── Step 1: Ingesta ───────────────────────────────────────────────────────────
step_ingesta = ProcessingStep(
    name='ingesta-guatape',
    processor=make_processor(),
    code=f'{SCRIPTS_PREFIX}/ingesta_guatape.py',
)
print('Step 1 definido: ingesta')

In [ ]:
# ── Step 2: Procesamiento ─────────────────────────────────────────────────────
step_procesamiento = ProcessingStep(
    name='procesamiento-guatape',
    processor=make_processor(),
    code=f'{SCRIPTS_PREFIX}/procesamiento_guatape.py',
    depends_on=[step_ingesta],
)
print('Step 2 definido: procesamiento')

In [ ]:
# ── Step 3: Entrenamiento ARIMA ───────────────────────────────────────────────
step_train_arima = ProcessingStep(
    name='entrenamiento-arima',
    processor=make_processor(),
    code=f'{SCRIPTS_PREFIX}/entrenamiento_arima.py',
    inputs=[
        ProcessingInput(
            source=CURATED_S3,
            destination='/opt/ml/processing/input/curated/embalse_guatape/',
        )
    ],
    outputs=[
        ProcessingOutput(
            source='/opt/ml/processing/output/model',
            destination=MODEL_ARIMA_S3,
            output_name='model-arima',
        )
    ],
    depends_on=[step_procesamiento],
)
print('Step 3 definido: entrenamiento ARIMA')

In [ ]:
# ── Step 4: Inferencia ARIMA ──────────────────────────────────────────────────
step_infer_arima = ProcessingStep(
    name='inferencia-arima',
    processor=make_processor(),
    code=f'{SCRIPTS_PREFIX}/inferencia_arima.py',
    inputs=[
        ProcessingInput(
            source=MODEL_ARIMA_S3,
            destination='/opt/ml/processing/input/model',
        )
    ],
    outputs=[
        ProcessingOutput(
            source='/opt/ml/processing/output/preds',
            destination=PREDS_ARIMA_S3,
            output_name='preds-arima',
        )
    ],
    depends_on=[step_train_arima],
)
print('Step 4 definido: inferencia ARIMA')

In [ ]:
# ── Step 5: Entrenamiento LSTM (incluye inferencia) ───────────────────────────
step_train_lstm = ProcessingStep(
    name='entrenamiento-lstm',
    processor=make_processor(),
    code=f'{SCRIPTS_PREFIX}/entrenamiento_lstm.py',
    depends_on=[step_procesamiento],
)
print('Step 5 definido: entrenamiento LSTM')

In [ ]:
# ── Ensamblar y subir pipeline ────────────────────────────────────────────────
pipeline = Pipeline(
    name=PIPELINE_NAME,
    steps=[
        step_ingesta,
        step_procesamiento,
        step_train_arima,
        step_infer_arima,
        step_train_lstm,
    ],
    sagemaker_session=session,
)

pipeline.upsert(role_arn=ROLE)
print(f'Pipeline "{PIPELINE_NAME}" registrado en SageMaker.')

In [ ]:
# ── Lanzar ejecución ──────────────────────────────────────────────────────────
execution = pipeline.start()
print(f'Ejecución iniciada: {execution.arn}')
print('Puedes monitorear el avance en:')
print(f'  SageMaker AI → Pipelines → {PIPELINE_NAME}')

In [ ]:
# ── Monitorear hasta completar (opcional, bloquea el kernel) ──────────────────
# Comenta esta celda si prefieres monitorear desde la consola
execution.wait()
steps_status = execution.list_steps()
for s in steps_status:
    print(f"{s['StepName']:30s} → {s['StepStatus']}")